In [1]:
!pip install yt-dlp pydub pandas pysrt torchaudio huggingface_hub coqui-tts

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.0/182.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 8.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.8/862.8 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.3/997.3 kB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 648.4/648.4 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 3.5 MB/s eta 0:00:00
  Created wheel for pysrt: filename=pysrt-1.1.2-py3-none-any.whl size=13443 sha256=460ceb75481d8c9b3bbde4b80beb3afc80b

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
WANDB_API_KEY = user_secrets.get_secret("WANDB_API_KEY")
READ_HF_TOKEN = user_secrets.get_secret("READ_HF_TOKEN")

# Hugging Face

In [3]:
from huggingface_hub import HfApi, login
login(READ_HF_TOKEN)

In [4]:
# api = HfApi()

# repo_id = "Mohamed-Mohamed-Ibrahim/egyptian-voice-xtts"

# api.create_repo(repo_id=repo_id, exist_ok=True)

# Wandb

In [5]:
import os

os.environ["WANDB_MODE"] = "disabled"  # fallback if no key

import wandb

run_name = "Fine-tune-EGTTS-8"
run = wandb.init(
    project="Fine-tune-XTTSv2",
    name=run_name,
    mode=os.environ["WANDB_MODE"]  # ensures offline if key missing
)

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [6]:
# run_name = "Fine-tune-EGTTS-9"
# wandb.init(
#     project="Fine-tune-XTTSv2",
#     name=run_name,
# )

# Data preparation

In [7]:
# import os
# import yt_dlp
# import pysrt
# import pandas as pd
# import re
# from pydub import AudioSegment

# # ==========================================
# # 1. PATHS & CONFIGURATION
# # ==========================================
# VIDEO_URL = "https://www.youtube.com/watch?v=FEdWNodbw8c"
# SRT_PATH = "/kaggle/input/solvy-resources/NoteGPT_ORIGINAL_TRANSCRIPT_1769599129860.srt"
# PROJECT_DIR = "xtts_voice_clone_project"
# DATASET_DIR = os.path.join(PROJECT_DIR, "dataset")
# WAV_DIR = os.path.join(DATASET_DIR, "wavs")
# CHECKPOINT_DIR = "/kaggle/working/XTTS-v2" # Must contain config.json and model.pth
# OUTPUT_DIR = os.path.join(PROJECT_DIR, "output")
# LANGUAGE_CODE = "ar"       # Explicitly set your language
# ARGET_SAMPLE_RATE = 22050

# os.makedirs(WAV_DIR, exist_ok=True)

# # ==========================================
# # 2. DATASET PREPARATION (Youtube + SRT)
# # ==========================================
# def prepare_data():
#     # print("--- Downloading Audio from YouTube ---")
#     audio_temp = os.path.join(PROJECT_DIR, "full_audio.wav")
#     ydl_opts = {
#         'format': 'bestaudio/best',
#         'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'wav', 'preferredquality': '192'}],
#         'outtmpl': audio_temp.replace(".wav", ""),
#     }
#     # ydl_opts = {
#     #     'format': 'bestaudio/best',
#     #     # --- THE COOKIE FIX ---
#     #     'cookiefile': 'youtube_cookies.txt', # Path to your exported cookies
#     #     # ----------------------
#     #     'postprocessors': [{
#     #         'key': 'FFmpegExtractAudio',
#     #         'preferredcodec': 'wav',
#     #         'preferredquality': '192'
#     #     }],
#     #     'outtmpl': audio_temp.replace(".wav", ""),
#     # }

#     # 
    
#     with yt_dlp.YoutubeDL(ydl_opts) as ydl:
#         ydl.download([VIDEO_URL])

#     print("--- Slicing Audio using SRT ---")
#     subs = pysrt.open(SRT_PATH, encoding='utf-8')
#     audio = AudioSegment.from_wav(audio_temp)
#     metadata = []
#     cnt = 0
#     for i, sub in enumerate(subs):
#         text = re.sub(r'<[^>]*>', '', sub.text).replace('\n', ' ').strip()
#         # Convert SRT time to ms
#         start_ms = (sub.start.hours*3600000 + sub.start.minutes*60000 + sub.start.seconds*1000 + sub.start.milliseconds)
#         end_ms = (sub.end.hours*3600000 + sub.end.minutes*60000 + sub.end.seconds*1000 + sub.end.milliseconds)
        
#         duration = (end_ms - start_ms) / 1000
#         if 2.0 <= duration <= 11.0 and text:
#             filename = f"sample_{i:04d}.wav"
#             chunk = audio[start_ms:end_ms].set_frame_rate(ARGET_SAMPLE_RATE).set_channels(1)
#             chunk.export(os.path.join(WAV_DIR, filename), format="wav")
#             metadata.append([f"waves/{filename}", text, "me"])
#             cnt += 1
#             if cnt >= 10:
#                 break
#     pd.DataFrame(metadata).to_csv(os.path.join(DATASET_DIR, "metadata.csv"), sep="|", index=False, header=False)
#     print(f"Dataset ready: {len(metadata)} samples.")


# if __name__ == "__main__":
#     prepare_data()
#     #

In [8]:
# !rm -rf /kaggle/working/xtts_voice_clone_project/dataset/wavs /kaggle/working/xtts_voice_clone_project/dataset/metadata.csv

# Fine tuning

In [9]:
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts
from trainer import Trainer, TrainerArgs
from TTS.tts.configs.shared_configs import BaseDatasetConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.layers.xtts.trainer.gpt_trainer import GPTTrainer, GPTTrainerConfig


2026-02-07 14:24:09.720435: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770474250.140728      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770474250.262489      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770474251.264401      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770474251.264445      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770474251.264448      24 computation_placer.cc:177] computation placer alr

In [10]:
import os
import glob
import torch
from trainer import Trainer, TrainerArgs
from TTS.utils.manage import ModelManager
from TTS.tts.layers.xtts.trainer.gpt_trainer import GPTTrainer, GPTTrainerConfig, GPTArgs
from TTS.tts.datasets import load_tts_samples
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.models.xtts import XttsAudioConfig

# ==========================================
# 1. PATHS & LOGGING
# ==========================================
RUN_NAME = "Egyptian_English_Refined_v1"
PROJECT_NAME = "Egyptian_TTS_Research"
DASHBOARD_LOGGER = "wandb" 
# OUT_PATH = './training_output/'
OUT_PATH = '/training_output/'
os.makedirs(OUT_PATH, exist_ok=True)

# Dataset Paths - Pointing to your Kaggle Input
DATASET_DIR = "/kaggle/input/solvy-audio-arabic/dataset"
WAV_DIR = os.path.join(DATASET_DIR, "wavs")

# Checkpoint Paths
CHECKPOINTS_OUT_PATH = os.path.join(OUT_PATH, "XTTS_v2_base_files/")
# CHECKPOINTS_OUT_PATH = os.path.join(OUT_PATH, "/EGTTS-V0.1_base_files/")
os.makedirs(CHECKPOINTS_OUT_PATH, exist_ok=True)

# ==========================================
# 2. MODEL DOWNLOADER & DYNAMIC REFERENCE
# ==========================================
DVAE_LINK = "https://coqui.gateway.scarf.sh/hf-coqui/XTTS-v2/main/dvae.pth"
MEL_NORM_LINK = "https://coqui.gateway.scarf.sh/hf-coqui/XTTS-v2/main/mel_stats.pth"
# TOKEN_LINK = "https://coqui.gateway.scarf.sh/hf-coqui/XTTS-v2/main/vocab.json"
TOKEN_LINK = "https://huggingface.co/OmarSamir/EGTTS-V0.1/resolve/main/vocab.json"
# MODEL_LINK = "https://coqui.gateway.scarf.sh/hf-coqui/XTTS-v2/main/model.pth"
# MODEL_LINK = "https://huggingface.co/OmarSamir/EGTTS-V0.1/resolve/main/model.pth"
MODEL_LINK = "https://huggingface.co/Mohamed-Mohamed-Ibrahim/Alaa-Nabel-XTTS-v2/resolve/main/model.pth"

DVAE_CHECKPOINT = os.path.join(CHECKPOINTS_OUT_PATH, "dvae.pth")
MEL_NORM_FILE = os.path.join(CHECKPOINTS_OUT_PATH, "mel_stats.pth")
TOKENIZER_FILE = os.path.join(CHECKPOINTS_OUT_PATH, "vocab.json")
XTTS_CHECKPOINT = os.path.join(CHECKPOINTS_OUT_PATH, "model.pth")

# if not os.path.isfile(XTTS_CHECKPOINT):
#     ModelManager._download_model_files([DVAE_LINK, MEL_NORM_LINK, TOKEN_LINK, MODEL_LINK], CHECKPOINTS_OUT_PATH, progress_bar=True)

SPEAKER_REFERENCE = ["/kaggle/input/solvy-audio-arabic/dataset/wavs/sample_0002.wav"]


In [11]:
!du -ah / | sort -rh | head -n 10

du: cannot access '/proc/90/task/90/fd/3': No such file or directory
du: cannot access '/proc/90/task/90/fdinfo/3': No such file or directory
du: cannot access '/proc/90/fd/4': No such file or directory
du: cannot access '/proc/90/fdinfo/4': No such file or directory
47G	/
35G	/usr
26G	/usr/local
20G	/usr/local/lib
19G	/usr/local/lib/python3.12/dist-packages
19G	/usr/local/lib/python3.12
8.3G	/root
5.5G	/usr/lib
4.8G	/usr/local/cuda-12.5
4.6G	/usr/lib/x86_64-linux-gnu
sort: write failed: 'standard output': Broken pipe
sort: write error


In [12]:
%%writefile train_xtts.py
import os
import glob
import torch
from trainer import Trainer, TrainerArgs
from TTS.utils.manage import ModelManager
from TTS.tts.layers.xtts.trainer.gpt_trainer import GPTTrainer, GPTTrainerConfig, GPTArgs
from TTS.tts.datasets import load_tts_samples
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.models.xtts import XttsAudioConfig

os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_SILENT"] = "true"

# ==========================================
# 1. PATHS & LOGGING
# ==========================================
RUN_NAME = "Egyptian_English_Refined_v1"
PROJECT_NAME = "Egyptian_TTS_Research"
DASHBOARD_LOGGER = "wandb" 
# OUT_PATH = './training_output/'
OUT_PATH = '/training_output/'
os.makedirs(OUT_PATH, exist_ok=True)

# Dataset Paths - Pointing to your Kaggle Input
DATASET_DIR = "/kaggle/input/fine-tune-egtts-v2-data-preparation-3/egyptian_dataset/"
WAV_DIR = os.path.join(DATASET_DIR, "wavs")

# Checkpoint Paths
# CHECKPOINTS_OUT_PATH = os.path.join(OUT_PATH, "XTTS_v2_base_files/")
CHECKPOINTS_OUT_PATH = os.path.join(OUT_PATH, "/XTTS_v2_base_files/")
os.makedirs(CHECKPOINTS_OUT_PATH, exist_ok=True)

# ==========================================
# 2. MODEL DOWNLOADER & DYNAMIC REFERENCE
# ==========================================
DVAE_LINK = "https://coqui.gateway.scarf.sh/hf-coqui/XTTS-v2/main/dvae.pth"
MEL_NORM_LINK = "https://coqui.gateway.scarf.sh/hf-coqui/XTTS-v2/main/mel_stats.pth"
# TOKEN_LINK = "https://coqui.gateway.scarf.sh/hf-coqui/XTTS-v2/main/vocab.json"
TOKEN_LINK = "https://huggingface.co/OmarSamir/EGTTS-V0.1/resolve/main/vocab.json"
# MODEL_LINK = "https://coqui.gateway.scarf.sh/hf-coqui/XTTS-v2/main/model.pth"
MODEL_LINK = "https://huggingface.co/OmarSamir/EGTTS-V0.1/resolve/main/model.pth"
# MODEL_LINK = "https://huggingface.co/Mohamed-Mohamed-Ibrahim/Alaa-Nabel-XTTS-v2/resolve/main/model.pth"

DVAE_CHECKPOINT = os.path.join(CHECKPOINTS_OUT_PATH, "dvae.pth")
MEL_NORM_FILE = os.path.join(CHECKPOINTS_OUT_PATH, "mel_stats.pth")
TOKENIZER_FILE = os.path.join(CHECKPOINTS_OUT_PATH, "vocab.json")
XTTS_CHECKPOINT = os.path.join(CHECKPOINTS_OUT_PATH, "model.pth")

if not os.path.isfile(XTTS_CHECKPOINT):
    ModelManager._download_model_files([DVAE_LINK, MEL_NORM_LINK, TOKEN_LINK, MODEL_LINK], CHECKPOINTS_OUT_PATH, progress_bar=True)

SPEAKER_REFERENCE = ["/kaggle/input/solvy-audio-arabic/dataset/wavs/sample_0002.wav"]

# ==========================================
# 3. HYPERPARAMETER TUNING (1.9k Samples / 30GB VRAM)
# ==========================================
BATCH_SIZE = 4           # Optimized for 30GB VRAM
GRAD_ACUMM_STEPS = 8    # (4 * 63 = 252) Target effective batch for stability

model_args = GPTArgs(
    max_conditioning_length=132300, 
    min_conditioning_length=66150,  
    max_wav_length=255995,          # ~11.6s to capture expressive Egyptian prosody
    max_text_length=250, 
    mel_norm_file=MEL_NORM_FILE,
    dvae_checkpoint=DVAE_CHECKPOINT,
    xtts_checkpoint=XTTS_CHECKPOINT,  
    tokenizer_file=TOKENIZER_FILE,
    gpt_num_audio_tokens=1026, 
    gpt_start_audio_token=1024,
    gpt_stop_audio_token=1025,
    gpt_use_masking_gt_prompt_approach=True,
    gpt_use_perceiver_resampler=True,
)

# Crucial: 22050Hz for XTTS v2 internal architecture
audio_config = XttsAudioConfig(sample_rate=22050, dvae_sample_rate=22050, output_sample_rate=24000) 

dataset_config = BaseDatasetConfig(
    formatter="ljspeech",
    dataset_name="egyptian_voice_ds",
    path=DATASET_DIR,
    meta_file_train=os.path.join(DATASET_DIR, "metadata.csv"),
)

config = GPTTrainerConfig(
    run_eval=True,
    epochs=7,                # Perfect for 1.9k samples
    output_path=OUT_PATH,
    model_args=model_args,
    run_name=RUN_NAME,
    project_name=PROJECT_NAME,
    # dashboard_logger= DASHBOARD_LOGGER,
    # log_model_step=None,
    audio=audio_config,
    batch_size=BATCH_SIZE,   
    eval_batch_size=2,
    num_loader_workers=4,
    # save_step= 10,    #20000       
    # save_n_checkpoints=2,
    # save_checkpoints=True,
    # --- LOGGING & DISK SAFETY ---
    # dashboard_logger=None,    # Set to None to disable WandB and Tensorboard
    logger_uri=None,          # Ensure no remote logging URI is set
    log_model_step=None,      # Disable periodic model logging to the dashboard
    

    # --- DISK OPTIMIZATION STARTS HERE ---
    save_step=20000,            # Frequent saves but we limit 'n'
    save_n_checkpoints=1,     # Only keep the latest to save space
    save_checkpoints=True,
    # save_all_best=False,      # Crucial: prevents saving multiple "best" files
    # -------------------------------------
    
    optimizer="AdamW",
    # mixed_precision=True,
    optimizer_params={"betas": [0.9, 0.96], "eps": 1e-8, "weight_decay": 1e-2},
    lr=5e-06,                # Stable refinement rate
    lr_scheduler="ExponentialLR", 
    lr_scheduler_params={"gamma": 0.95},
    test_sentences=[ 
        {"text": "أهلاً بك، أنا الآن أتحدث بصوتي الخاص بعد التدريب.", "language": "ar", "speaker_wav": SPEAKER_REFERENCE},
        {"text": "اهلا بيك يا صحبي نورت الدنيا كلها.", "language": "ar", "speaker_wav": SPEAKER_REFERENCE},
        {"text": "Developing this RAG system for Egyptian students is my main goal.", "language": "en", "speaker_wav": SPEAKER_REFERENCE}
    ],
)

# ==========================================
# 4. EXECUTION
# ==========================================
train_samples, eval_samples = load_tts_samples(
    dataset_config, 
    eval_split=True, 
    eval_split_max_size=256, 
    eval_split_size=0.01
)

# Fix for the "NotImplementedError: Language ''"
for sample in train_samples:
    sample["language"] = "ar"
for sample in eval_samples:
    sample["language"] = "ar"

model = GPTTrainer.init_from_config(config)

trainer = Trainer(
    TrainerArgs(
        restore_path=None,
        skip_train_epoch=False,
        start_with_eval=False,
        grad_accum_steps=GRAD_ACUMM_STEPS,
    ),
    config,
    output_path=OUT_PATH,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)

trainer.fit()

Writing train_xtts.py


In [13]:
!python3 -m trainer.distribute --gpus "0,1" --script train_xtts.py

['train_xtts.py', '--continue_path=', '--restore_path=', '--group_id=group_2026_02_07-142615', '--use_ddp=true', '--rank=0']
['train_xtts.py', '--continue_path=', '--restore_path=', '--group_id=group_2026_02_07-142615', '--use_ddp=true', '--rank=1']
2026-02-07 14:26:25.199038: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-07 14:26:25.199094: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770474385.221569      98 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770474385.221827      97 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has alre

In [14]:
!cp $TOKENIZER_FILE /training_output/

cp: cannot stat '/training_output/XTTS_v2_base_files/vocab.json': No such file or directory


In [15]:
!du -ah / | sort -rh | head -n 10

du: cannot access '/proc/393/task/393/fd/3': No such file or directory
du: cannot access '/proc/393/task/393/fdinfo/3': No such file or directory
du: cannot access '/proc/393/fd/4': No such file or directory
du: cannot access '/proc/393/fdinfo/4': No such file or directory
63G	/
35G	/usr
26G	/usr/local
20G	/usr/local/lib
19G	/usr/local/lib/python3.12/dist-packages
19G	/usr/local/lib/python3.12
11G	/training_output/Egyptian_English_Refined_v1-February-07-2026_02+28PM-0000000
11G	/training_output
8.3G	/root
5.5G	/XTTS_v2_base_files
sort: write failed: 'standard output': Broken pipe
sort: write error


In [16]:
!zip -r submission.zip /training_output

  adding: training_output/ (stored 0%)
  adding: training_output/XTTS_v2_base_files/ (stored 0%)
  adding: training_output/Egyptian_English_Refined_v1-February-07-2026_02+28PM-0000000/ (stored 0%)
  adding: training_output/Egyptian_English_Refined_v1-February-07-2026_02+28PM-0000000/train_xtts.py (deflated 61%)
  adding: training_output/Egyptian_English_Refined_v1-February-07-2026_02+28PM-0000000/config.json (deflated 70%)
  adding: training_output/Egyptian_English_Refined_v1-February-07-2026_02+28PM-0000000/best_model.pth (deflated 9%)
  adding: training_output/Egyptian_English_Refined_v1-February-07-2026_02+28PM-0000000/events.out.tfevents.1770474494.9e96bed0d1ea.97.0 (deflated 14%)
  adding: training_output/Egyptian_English_Refined_v1-February-07-2026_02+28PM-0000000/trainer_0_log.txt (deflated 73%)
  adding: training_output/Egyptian_English_Refined_v1-February-07-2026_02+28PM-0000000/best_model_973.pth (deflated 9%)
  adding: training_output/Egyptian_English_Refined_v1-February-07-

In [17]:
# from TTS.tts.configs.xtts_config import XttsConfig
# from TTS.tts.models.xtts import Xtts

# config = XttsConfig()
# config.load_json("/training_output/Egyptian_English_Refined_v1-January-29-2026_07+18PM-0000000/config.json")
# model = Xtts.init_from_config(config)
# model.load_checkpoint(config, checkpoint_path="/training_output/Egyptian_English_Refined_v1-January-29-2026_07+18PM-0000000/best_model.pth", eval=True)
# model.cuda()

# outputs = model.synthesize(
#     "الجملة أسلوب شرط جازم، وفيها الأفعال المضارعة مجزومة: يَرَ فعل مضارع مجزوم وعلامة جزمه حذف حرف العلة لأنه معتل الآخر، وتكسبوا فعل مضارع مجزوم وعلامة جزمه حذف النون لأنه من الأفعال الخمسة وواو الجماعة ضمير متصل في محل رفع فاعل.",
#     # "The quick brown fox jumps over the lazy dog. Heavy, dark clouds hung over the valley as the thin, silver moon rose. Please call Stella and ask her to bring these things from the store: six spoons of fresh snow peas, and maybe a snack for the brother.",
#     # "أهلاً بيك يا صاحبي، نورت الدنيا كلها. و يا باشا خلاص يا باشا يا باشا لا  يا باشا.",
#     config,
#     speaker_wav="/kaggle/input/solvy-audio-arabic-10/dataset_10/wavs/sample_0002.wav",
#     gpt_cond_len=3,
#     language="ar",
#     # language="en",
# )

In [18]:
# from IPython.display import Audio, display
# import torchaudio

# AUDIO_OUTPUT_PATH = "/kaggle/working/output_audio.wav"
# torchaudio.save(AUDIO_OUTPUT_PATH, torch.tensor(outputs["wav"]).unsqueeze(0), 24000)
# display(Audio(AUDIO_OUTPUT_PATH, autoplay=True))

In [19]:
!ls /training_output/

Egyptian_English_Refined_v1-February-07-2026_02+28PM-0000000
XTTS_v2_base_files
